<a href="https://colab.research.google.com/github/Dheepthi-Reddy/Paper-Implementations/blob/main/GPT/GPT1_From_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GPT-1: Improving Language Understanding by Generative Pre-Training — From Scratch

### A scaled-down PyTorch implementation following Radford et al., OpenAI 2018

This notebook implements GPT-1 from scratch following the original paper closely.

**What the original paper did:**
- Decoder-only Transformer (12 layers, 768 hidden, 12 heads)
- Pre-trained on BooksCorpus (800M words) using next word prediction
- Fine-tuned on 9 NLP tasks with task-aware input transformations
- Analyzed layer transfer and zero-shot behavior

**What I did:**
- Same decoder-only architecture, scaled down
- Pre-trained on WikiText-103 using next word prediction
- Fine-tuned on SST-2 sentiment classification
- Layer transfer experiment mirroring Figure 2 left
- Zero-shot evaluation mirroring Figure 2 right

- GPT uses **decoder only** — no encoder, no cross-attention
- GPT uses **causal (left-to-right) mask** — each token only sees previous tokens
- GPT pre-trains with **next word prediction** — not MLM or NSP
- GPT uses **post-layer normalization** — LayerNorm after attention (same as original Transformer)
- GPT uses **no segment embeddings** — only token + position embeddings

**Scaled-down config:**

| Parameter | Paper (GPT-1) | This notebook |
|---|---|---|
| Layers | 12 | 4 |
| Hidden size | 768 | 256 |
| Attention heads | 12 | 4 |
| Feed-forward dim | 3072 | 512 |
| Dataset | BooksCorpus (800M words) | WikiText-103 |
| Tokenizer | BPE (40K vocab) | Word-level (30K vocab) |
| Parameters | 117M | ~12M |

---
## Step 0 — Enable GPU
Go to **Runtime → Change runtime type → GPU (T4)** before running.


In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

CUDA available: True
Device: Tesla T4


---
## Step 1 — Install dependencies


In [2]:
!pip install -q datasets scikit-learn tokenizers

---
## Step 2 — GPTConfig

Mirroring the paper's model specification. All hyperparameters in one place.
but added one here for clean coding.


In [3]:
class GPTConfig:

    def __init__(
        self,
        vocab_size,
        n_embd=256,           # hidden size — called n_embd in GPT terminology
        n_layer=4,            # number of decoder layers
        n_head=4,             # number of attention heads
        n_ff=512,             # feed-forward dimension (4 * n_embd in paper)
        block_size=128,       # maximum sequence length
        dropout=0.1,
    ):
        self.vocab_size = vocab_size
        self.n_embd = n_embd
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_ff = n_ff
        self.block_size = block_size
        self.dropout = dropout
        assert n_embd % n_head == 0, f"n_embd ({n_embd}) must be divisible by n_head ({n_head})"

    def __repr__(self):
        return (f"GPTConfig(n_embd={self.n_embd}, n_layer={self.n_layer}, "
                f"n_head={self.n_head}, n_ff={self.n_ff}, block_size={self.block_size})")

print("Defined GPTConfig")

Defined GPTConfig


---
## Step 3 — Build BPE Tokenizer

The original GPT-1 paper trained a **Byte Pair Encoding (BPE)** tokenizer on BooksCorpus with 40,000 merge operations. BooksCorpus is not available publicly so we train our own BPE tokenizer on WikiText-2 using the same approach.

**Why BPE instead of word tokenizer:**
- No `[UNK]` tokens — any word can be represented as subpieces
- Better vocabulary coverage — 30K subword tokens cover far more words than 30K whole words
- Matches the paper's actual implementation

**Special tokens matching the paper exactly:**
- `<start>` — beginning of every sequence
- `<extract>` — end of sequence, used for classification
- `<delim>` — separator between sentences for pair tasks
- `<pad>` — padding token (needed for batching)


In [4]:
from datasets import load_dataset
from tokenizers import ByteLevelBPETokenizer
import os

raw_dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

# Collect all training text
print("Collecting text for BPE training...")
train_texts = [ex["text"] for ex in raw_dataset["train"] if len(ex["text"].strip()) > 10]
print(f"Total training texts: {len(train_texts)}")

# Save texts temporarily for BPE training
os.makedirs("/tmp/gpt_bpe", exist_ok=True)
with open("/tmp/gpt_bpe/train.txt", "w") as f:
    f.write("\n".join(train_texts))

# Train BPE tokenizer — same approach as the paper
print("Training BPE tokenizer...")
tokenizer = ByteLevelBPETokenizer()
tokenizer.train(
    files=["/tmp/gpt_bpe/train.txt"],
    vocab_size=30000,
    min_frequency=2,
    special_tokens=["<pad>", "<unk>", "<start>", "<extract>", "<delim>"]
)

# Save tokenizer
tokenizer.save_model("/tmp/gpt_bpe")
print("BPE tokenizer trained and saved")

# Special token IDs
PAD_IDX     = tokenizer.token_to_id("<pad>")
UNK_IDX     = tokenizer.token_to_id("<unk>")
START_IDX   = tokenizer.token_to_id("<start>")
EXTRACT_IDX = tokenizer.token_to_id("<extract>")
DELIM_IDX   = tokenizer.token_to_id("<delim>")

VOCAB_SIZE = tokenizer.get_vocab_size()
print(f"\nVocabulary size: {VOCAB_SIZE}")
print(f"Special tokens (matching paper):")
print(f"  <pad>:     {PAD_IDX}")
print(f"  <start>:   {START_IDX}")
print(f"  <extract>: {EXTRACT_IDX}")
print(f"  <delim>:   {DELIM_IDX}")

# Test tokenization
sample = "The Transformer model has revolutionized natural language processing"
encoded = tokenizer.encode(sample)
print(f"\nSample: {sample}")
print(f"Tokens: {encoded.tokens}")
print(f"IDs:    {encoded.ids}")
print(f"\nNote: No [UNK] tokens — BPE handles all words through subpieces")

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

...

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Total training texts: 23547
Training BPE tokenizer...
BPE tokenizer trained and saved

Vocabulary size: 30000
Special tokens (matching paper):
  <pad>:     0
  <start>:   2
  <extract>: 3
  <delim>:   4

Sample: The Transformer model has revolutionized natural language processing
Tokens: ['T', 'he', 'ĠTrans', 'form', 'er', 'Ġmodel', 'Ġhas', 'Ġrevolution', 'ized', 'Ġnatural', 'Ġlanguage', 'Ġprocessing']
IDs:    [56, 262, 3811, 718, 266, 4249, 566, 6323, 1034, 3578, 3033, 11612]

Note: No [UNK] tokens — BPE handles all words through subpieces


---
## Step 4 — GPT pre-training dataset

GPT pre-trains with **next word prediction** (language modeling).

GPT simply predicts the next token at every position.

Input:  `[BOS] the dog ran in the`
Target: `the dog ran in the park`

Every input token predicts the next one. No masking needed.
The causal attention mask ensures each position only sees previous tokens.


In [5]:
import torch
from torch.utils.data import Dataset, DataLoader

def get_text_chunks(dataset, block_size=128, max_examples=None):
    """
    Concatenate all text into one long sequence of BPE tokens and split into chunks.
    This is how GPT-1 pre-training data was prepared — long contiguous sequences
    that allow the model to learn long-range dependencies.

    """
    all_ids = []
    examples = list(dataset)[:max_examples] if max_examples else list(dataset)

    for example in examples:
        text = example["text"].strip()
        if len(text) < 10:
            continue
        encoded = tokenizer.encode(text)
        all_ids.extend(encoded.ids)

    chunks = []
    for i in range(0, len(all_ids) - block_size, block_size):
        chunk = all_ids[i:i + block_size + 1]
        chunks.append(chunk)
    return chunks

class GPTDataset(Dataset):
    """
    Each item is a chunk of BPE tokens.
    Input: chunk[:-1] (all but last token)
    Target: chunk[1:]  (all but first token — shifted by one)

    Pure language modeling — predict every next token.
    Causal attention handles the left-to-right constraint.
    """
    def __init__(self, chunks):
        self.chunks = chunks

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        chunk = self.chunks[idx]
        x = torch.tensor(chunk[:-1], dtype=torch.long)
        y = torch.tensor(chunk[1:],  dtype=torch.long)
        return x, y

print("Building dataset chunks with BPE tokenizer...")
train_chunks = get_text_chunks(raw_dataset["train"], block_size=128, max_examples=5000)
valid_chunks = get_text_chunks(raw_dataset["validation"], block_size=128, max_examples=500)

train_ds = GPTDataset(train_chunks)
valid_ds = GPTDataset(valid_chunks)

BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                         generator=torch.Generator().manual_seed(42))
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train chunks: {len(train_ds)} | Valid chunks: {len(valid_ds)}")
print(f"Train batches: {len(train_loader)} | Valid batches: {len(valid_loader)}")

x_sample, y_sample = next(iter(train_loader))
print(f"Input shape: {x_sample.shape} | Target shape: {y_sample.shape}")

Building dataset chunks with BPE tokenizer...
Train chunks: 2333 | Valid chunks: 220
Train batches: 37 | Valid batches: 4
Input shape: torch.Size([64, 128]) | Target shape: torch.Size([64, 128])


---
## Step 5 — GPT Architecture


1. **No encoder** — GPT is decoder only
2. **Causal mask always on** — every layer uses left-to-right masking
3. **No cross-attention** — removed entirely since there is no encoder output
4. **Post-layer normalization** — LayerNorm after attention and feed-forward (same as original Transformer)
5. **No segment embeddings** — only token + position embeddings

### 5.1 — Causal Self-Attention


In [6]:
import torch.nn as nn
import math

class CausalSelfAttention(nn.Module):
    """
    Self-attention with causal (left-to-right) mask.

    Every token can only attend to itself and previous tokens.
    This is what makes GPT generative — it never sees future tokens.

    GPT uses causal attention — each token only sees previous tokens.
    GPT uses causal attention throughout all layers.
    """
    def __init__(self, config):
        super().__init__()
        self.n_head = config.n_head
        self.d_k = config.n_embd // config.n_head

        self.w_q = nn.Linear(config.n_embd, config.n_embd)
        self.w_k = nn.Linear(config.n_embd, config.n_embd)
        self.w_v = nn.Linear(config.n_embd, config.n_embd)
        self.w_o = nn.Linear(config.n_embd, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)

        # Register causal mask as a buffer — not a parameter, but moves to GPU automatically
        # Lower triangular matrix: position i can attend to positions 0..i
        self.register_buffer(
            "causal_mask",
            torch.tril(torch.ones(config.block_size, config.block_size))
        )

    def split_heads(self, x):
        B, T, D = x.size()
        return x.view(B, T, self.n_head, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        B, H, T, dk = x.size()
        return x.transpose(1, 2).contiguous().view(B, T, H * dk)

    def forward(self, x):
        B, T, D = x.size()
        q = self.split_heads(self.w_q(x))
        k = self.split_heads(self.w_k(x))
        v = self.split_heads(self.w_v(x))

        # Scaled dot-product attention
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)

        # Apply causal mask — mask out future positions
        mask = self.causal_mask[:T, :T].unsqueeze(0).unsqueeze(1)
        scores = scores.masked_fill(mask == 0, float("-1e9"))

        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, v)
        return self.w_o(self.combine_heads(out))

print("CausalSelfAttention defined")

CausalSelfAttention defined


In [7]:
class FeedForward(nn.Module):
    """
    Position-wise feed-forward network.
    GPT-1 uses GELU activation ().
    The paper uses 4x expansion: n_embd -> 4*n_embd -> n_embd
    """
    def __init__(self, config):
        super().__init__()
        self.linear1 = nn.Linear(config.n_embd, config.n_ff)
        self.linear2 = nn.Linear(config.n_ff, config.n_embd)
        self.gelu = nn.GELU()
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.dropout(self.linear2(self.gelu(self.linear1(x))))

print("Defined FeedForward")

Defined FeedForward


### 5.3 — GPT Decoder Block

**Post-layer normalization** (same as original Transformer paper):
- Attention → Add → LayerNorm
- FeedForward → Add → LayerNorm

This is different from many modern implementations which use pre-norm.
I followed the original GPT-1 paper exactly.


In [8]:
class GPTBlock(nn.Module):
    """
    Single GPT decoder block.

    Structure (post-norm, following original GPT-1 paper):
    x -> CausalSelfAttention -> x + residual -> LayerNorm -> FeedForward -> x + residual -> LayerNorm

    Post-norm: LayerNorm after attention and feed-forward.
    """
    def __init__(self, config):
        super().__init__()
        self.attention = CausalSelfAttention(config)
        self.feed_forward = FeedForward(config)
        self.norm1 = nn.LayerNorm(config.n_embd)
        self.norm2 = nn.LayerNorm(config.n_embd)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        # Self-attention + residual + LayerNorm (post-norm)
        x = self.norm1(x + self.dropout(self.attention(x)))
        # Feed-forward + residual + LayerNorm (post-norm)
        x = self.norm2(x + self.dropout(self.feed_forward(x)))
        return x

print("Defined GPTBlock")

Defined GPTBlock


### 5.4 — Full GPT Model

- Token embedding — what word is this?

**No segment embedding** — GPT does not distinguish sentence A from sentence B during pre-training.

**Two heads:**
- LM head — predicts next token (used during pre-training)
- Classification head — predicts class label (added during fine-tuning)


In [9]:
class GPT(nn.Module):
    """
    Full GPT-1 model following Radford et al., 2018.

    Decoder-only Transformer with:
    - Causal self-attention (left-to-right only)
    - Token + Position embeddings (no segment embedding)
    - Post-layer normalization
    - GELU activation
    - Language modeling head for pre-training
    - Classification head for fine-tuning
    """
    def __init__(self, config):
        super().__init__()
        self.config = config

        # Embeddings — token + position only (no segment embedding)
        self.token_embedding    = nn.Embedding(config.vocab_size, config.n_embd)
        self.position_embedding = nn.Embedding(config.block_size, config.n_embd)
        self.dropout            = nn.Dropout(config.dropout)

        # Stack of decoder blocks
        self.blocks = nn.ModuleList([GPTBlock(config) for _ in range(config.n_layer)])

        # Language modeling head — predicts next token
        # Weight tying: share weights between token embedding and LM head
        # This is used in the original GPT-1 paper and reduces parameters
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight  # weight tying

        # Initialize weights
        self.apply(self._init_weights)

    def _init_weights(self, module):
        """Initializing the weights — normal distribution with small std."""
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx):
        """
        Forward pass for pre-training (language modeling).
        idx: (batch, seq_len) token indices
        returns: (batch, seq_len, vocab_size) logits for next token prediction
        """
        B, T = idx.size()
        assert T <= self.config.block_size, f"Sequence length {T} exceeds block_size {self.config.block_size}"

        # Creating position indices
        positions = torch.arange(T, device=idx.device).unsqueeze(0).expand(B, T)

        # Sum token and position embeddings (no segment embedding)
        x = self.dropout(self.token_embedding(idx) + self.position_embedding(positions))

        # Pass through all decoder blocks
        for block in self.blocks:
            x = block(x)

        # LM head — project to vocabulary size
        logits = self.lm_head(x)
        return logits, x  # return both logits and hidden states for analysis

    def get_hidden_states(self, idx, n_layers=None):
        """
        Get hidden states after transferring n_layers.
        Used for layer transfer experiment.
        If n_layers=None, use all layers.
        """
        B, T = idx.size()
        positions = torch.arange(T, device=idx.device).unsqueeze(0).expand(B, T)
        x = self.dropout(self.token_embedding(idx) + self.position_embedding(positions))

        layers_to_use = self.blocks[:n_layers] if n_layers is not None else self.blocks
        for block in layers_to_use:
            x = block(x)
        return x

print("Defined GPT model")

Defined GPT model


---
## Step 6 — Instantiate model


In [10]:
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

config = GPTConfig(
    vocab_size=VOCAB_SIZE,
    n_embd=256,
    n_layer=4,
    n_head=4,
    n_ff=512,
    block_size=128,
    dropout=0.1
)

model = GPT(config).to(device)
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Config: {config}")
print(f"Total trainable parameters: {num_params:,} ({num_params/1e6:.1f}M)")
print(f"\nComparison:")
print(f"  GPT-1 paper:  117M parameters")
print(f"  This implemented GPT:      {num_params/1e6:.1f}M parameters")

Config: GPTConfig(n_embd=256, n_layer=4, n_head=4, n_ff=512, block_size=128)
Total trainable parameters: 9,821,184 (9.8M)

Comparison:
  GPT-1 paper:  117M parameters
  This implemented GPT:      9.8M parameters


---
## Step 7 — Loss function and optimizer

GPT-1 uses:
- Cross entropy loss on next token prediction
- Adam optimizer
- Gradient clipping



In [11]:
import time

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = torch.optim.Adam(model.parameters(), lr=2.5e-4, betas=(0.9, 0.999), eps=1e-8)

class WarmupScheduler:
    """
    Learning rate warmup scheduler from the paper.
    Linearly increases LR for warmup_steps then decays.
    """
    def __init__(self, optimizer, warmup_steps=200):
        self.optimizer = optimizer
        self.warmup_steps = warmup_steps
        self.step_num = 0

    def step(self):
        self.step_num += 1
        if self.step_num < self.warmup_steps:
            lr = 2.5e-4 * (self.step_num / self.warmup_steps)
        else:
            lr = 2.5e-4 * (self.warmup_steps / self.step_num) ** 0.5
        for pg in self.optimizer.param_groups:
            pg["lr"] = lr
        self.optimizer.step()

scheduler = WarmupScheduler(optimizer, warmup_steps=200)
print("Optimizer and scheduler ready")

print(f"Train batches per epoch: {len(train_loader)}")
print(f"Estimated time per epoch: ~{len(train_loader) * 0.5:.0f} seconds")

Optimizer and scheduler ready
Train batches per epoch: 37
Estimated time per epoch: ~18 seconds


---
## Step 8 — Pre-training loop

GPT pre-trains with pure language modeling — predict the next token at every position.

Saved checkpoints every 2 epochs for the zero-shot evaluation experiment.


In [12]:
def train_epoch(model, dataloader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits, _ = model(x)
        # Reshape for cross entropy: (B, T, V) -> (B*T, V) and (B, T) -> (B*T)
        loss = criterion(logits.view(-1, config.vocab_size), y.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            logits, _ = model(x)
            loss = criterion(logits.view(-1, config.vocab_size), y.view(-1))
            total_loss += loss.item()
    return total_loss / len(dataloader)

print("Training functions defined")

Training functions defined


In [13]:
NUM_EPOCHS = 10
best_valid_loss = float("inf")
checkpoints = {}  # store checkpoints at different epochs for zero-shot analysis

print(f"Pre-training GPT on WikiText-103 for {NUM_EPOCHS} epochs")
print(f"Config: {config.n_layer} layers, n_embd={config.n_embd}, heads={config.n_head}")
print("-" * 70)

for epoch in range(1, NUM_EPOCHS + 1):
    start = time.time()
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    valid_loss = evaluate(model, valid_loader, device)
    elapsed = time.time() - start

    # Save best checkpoint
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), "gpt_best.pt")

    # Save checkpoint every 2 epochs for zero-shot analysis
    if epoch % 2 == 0:
        torch.save(model.state_dict(), f"gpt_epoch_{epoch}.pt")
        checkpoints[epoch] = valid_loss
        print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.3f} | Val Loss: {valid_loss:.3f} | Time: {elapsed:.1f}s ← checkpoint saved")
    else:
        print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.3f} | Val Loss: {valid_loss:.3f} | Time: {elapsed:.1f}s")

Pre-training GPT on WikiText-103 for 10 epochs
Config: 4 layers, n_embd=256, heads=4
----------------------------------------------------------------------
Epoch 01 | Train Loss: 10.298 | Val Loss: 10.191 | Time: 7.2s
Epoch 02 | Train Loss: 9.660 | Val Loss: 9.114 | Time: 6.2s ← checkpoint saved
Epoch 03 | Train Loss: 8.519 | Val Loss: 8.091 | Time: 6.3s
Epoch 04 | Train Loss: 7.602 | Val Loss: 7.593 | Time: 6.5s ← checkpoint saved
Epoch 05 | Train Loss: 7.332 | Val Loss: 7.487 | Time: 6.4s
Epoch 06 | Train Loss: 7.149 | Val Loss: 7.295 | Time: 6.4s ← checkpoint saved
Epoch 07 | Train Loss: 6.929 | Val Loss: 7.113 | Time: 6.5s
Epoch 08 | Train Loss: 6.696 | Val Loss: 6.960 | Time: 6.5s ← checkpoint saved
Epoch 09 | Train Loss: 6.495 | Val Loss: 6.860 | Time: 6.6s
Epoch 10 | Train Loss: 6.336 | Val Loss: 6.790 | Time: 6.7s ← checkpoint saved


---
## Step 9 — Text generation test

Before fine-tuning, verifying if the pre-trained model learned something useful
by generating text from a prompt.


In [14]:
def generate(model, prompt, device, max_new_tokens=30, temperature=0.8):
    """
    Generate text autoregressively using BPE tokenizer.
    This is GPT's core capability.
    """
    model.eval()
    encoded = tokenizer.encode(prompt)
    tokens = [START_IDX] + encoded.ids
    input_ids = torch.tensor([tokens]).to(device)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            input_truncated = input_ids[:, -config.block_size:]
            logits, _ = model(input_truncated)
            next_logits = logits[0, -1, :] / temperature
            probs = torch.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)
            if next_token.item() == EXTRACT_IDX:
                break

    # Decode using BPE tokenizer
    generated_ids = input_ids[0].tolist()[1:]  # skip START token
    generated_text = tokenizer.decode(generated_ids)
    return generated_text

# Load best checkpoint
model.load_state_dict(torch.load("gpt_best.pt"))

prompts = [
    "the history of the united states",
    "scientists have discovered",
    "the best way to learn",
]

print("Text generation from pre-trained GPT (BPE tokenizer):")
print("-" * 60)
for prompt in prompts:
    generated = generate(model, prompt, device)
    print(f"Prompt:    {prompt}")
    print(f"Generated: {generated}")
    print()

Text generation from pre-trained GPT (BPE tokenizer):
------------------------------------------------------------
Prompt:    the history of the united states
Generated: the history of the united states in in the Review , New City of the British2 , and passing no Polish their 18 @.@ 2010 , and suffered replacement .= = = = =

Prompt:    scientists have discovered
Generated: scientists have discovered the black in the Egyptian seasonDTV . She was new Polish professional producers or the 1992 , about the Yorkal , Díaz . A 5 @-@ @-@ B

Prompt:    the best way to learn
Generated: the best way to learn sold are the victims ; the Human @-@ all in the lead in the Decgeon .The team was norms use of the te diverted to the consequences



---
## Step 10 — Zero-shot evaluation (Figure 2 right of the paper)

The paper found that even without fine-tuning, the pre-trained model shows
task performance that improves steadily with more pre-training steps.

Here I evaluated the model on SST-2 at different pre-training checkpoints
without any fine-tuning — just using the language model directly.

For zero-shot sentiment classification we use a simple approach:
given a sentence, compute the probability of "positive" vs "negative"
words appearing after it.


In [15]:
from datasets import load_dataset as hf_load

sst2 = hf_load("nyu-mll/glue", "sst2")

# Positive and negative probe words — encode with BPE
POSITIVE_WORDS = ["good", "great", "excellent", "wonderful", "amazing", "positive", "best"]
NEGATIVE_WORDS = ["bad", "terrible", "awful", "horrible", "poor", "negative", "worst"]

pos_ids = [tokenizer.encode(w).ids[0] for w in POSITIVE_WORDS]
neg_ids = [tokenizer.encode(w).ids[0] for w in NEGATIVE_WORDS]

def zero_shot_sentiment(model, sentence, device):
    """
    Zero-shot sentiment classification using BPE tokenizer.
    Encode sentence with paper's <start> token and check
    if positive or negative words have higher next-token probability.
    """
    model.eval()
    encoded = tokenizer.encode(sentence)
    tokens = [START_IDX] + encoded.ids[:config.block_size - 1]
    input_ids = torch.tensor([tokens]).to(device)

    with torch.no_grad():
        logits, _ = model(input_ids)
        probs = torch.softmax(logits[0, -1, :], dim=-1)

    pos_score = sum(probs[i].item() for i in pos_ids if i < len(probs))
    neg_score = sum(probs[i].item() for i in neg_ids if i < len(probs))
    return 1 if pos_score > neg_score else 0

def evaluate_zero_shot(model, dataset, device, n_samples=200):
    correct = total = 0
    for i, ex in enumerate(dataset):
        if i >= n_samples:
            break
        pred = zero_shot_sentiment(model, ex["sentence"], device)
        if pred == ex["label"]:
            correct += 1
        total += 1
    return correct / total

print("Zero-shot sentiment evaluation at different pre-training checkpoints:")
print("-" * 60)
zero_shot_results = {}

for epoch, val_loss in sorted(checkpoints.items()):
    model.load_state_dict(torch.load(f"gpt_epoch_{epoch}.pt"))
    acc = evaluate_zero_shot(model, sst2["validation"], device)
    zero_shot_results[epoch] = acc
    print(f"Epoch {epoch:2d} (Val Loss: {val_loss:.3f}) | Zero-shot SST-2: {acc:.3f}")

print()
print("Expected: accuracy should improve as pre-training continues")
print("This mirrors Figure 2 right of the paper")

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

sst2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

sst2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sst2/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 72.8kB            

sst2/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

sst2/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

Zero-shot sentiment evaluation at different pre-training checkpoints:
------------------------------------------------------------
Epoch  2 (Val Loss: 9.114) | Zero-shot SST-2: 0.495
Epoch  4 (Val Loss: 7.593) | Zero-shot SST-2: 0.495
Epoch  6 (Val Loss: 7.295) | Zero-shot SST-2: 0.495
Epoch  8 (Val Loss: 6.960) | Zero-shot SST-2: 0.495
Epoch 10 (Val Loss: 6.790) | Zero-shot SST-2: 0.495

Expected: accuracy should improve as pre-training continues
This mirrors Figure 2 right of the paper


---
## Step 11 — Fine-tuning on SST-2

GPT-1 fine-tuning uses task-aware input transformations from Figure 1 of the paper.

For classification (SST-2):
`[BOS] sentence [EOS] -> Transformer -> Linear`

The model reads the full sentence and uses the representation at the [EOS] token

**Why EOS instead of CLS?**
GPT reads left to right. The last token [EOS] has seen the entire sentence
through its causal attention, making it the richest representation.


**Auxiliary language modeling loss (Section 3.2 of paper):**
During fine-tuning the paper adds the language modeling objective as an auxiliary loss:

`Total loss = Classification loss + λ × Language Modeling loss`

where λ = 0.5. This helps the model retain its language understanding while specializing for the downstream task and improves generalization.


In [16]:
from torch.nn.utils.rnn import pad_sequence

class SST2Dataset(torch.utils.data.Dataset):
    """
    SST-2 dataset for GPT fine-tuning.

    Input format matching the paper (Figure 1, Classification):
    <start> sentence <extract>

    Classification uses representation at <extract> token position.
    Matching the paper's task-aware input transformation exactly.
    """
    def __init__(self, split, max_len=64):
        self.data = []
        for ex in split:
            # Encode sentence using BPE tokenizer
            encoded = tokenizer.encode(ex["sentence"])
            token_ids = encoded.ids[:max_len - 2]

            # Paper's format: <start> sentence <extract>
            ids = [START_IDX] + token_ids + [EXTRACT_IDX]

            self.data.append({
                "input_ids": torch.tensor(ids),
                "label": torch.tensor(ex["label"])
            })

    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]

def sst2_collate(batch):
    input_ids = pad_sequence([b["input_ids"] for b in batch], batch_first=True, padding_value=PAD_IDX)
    labels    = torch.stack([b["label"] for b in batch])
    return {"input_ids": input_ids, "labels": labels}

train_sst2 = torch.utils.data.DataLoader(SST2Dataset(sst2["train"]), batch_size=32, shuffle=True, collate_fn=sst2_collate)
valid_sst2 = torch.utils.data.DataLoader(SST2Dataset(sst2["validation"]), batch_size=32, collate_fn=sst2_collate)
print(f"SST-2 Train: {len(SST2Dataset(sst2['train']))} | Valid: {len(SST2Dataset(sst2['validation']))}")
print(f"Input format: <start> sentence <extract> (matching paper Figure 1)")

SST-2 Train: 67349 | Valid: 872
Input format: <start> sentence <extract> (matching paper Figure 1)


---
## Step 12 — Layer transfer experiment (Figure 2 left of the paper)

The paper tested: how many layers should you transfer from pre-training to fine-tuning?

I tested transferring 0 (embeddings only), 1, 2, 3, and 4 layers.
For each configuration freezing the transferred layers and only train a new classifier head.

Expected finding: more layers transferred = better performance.


In [17]:
class GPTClassifier(nn.Module):
    """
    GPT fine-tuned for classification.

    Uses representation at [EOS] token position for classification.
    This follows GPT-1's approach — the last token has seen everything.

    n_layers_to_use controls how many pre-trained layers to use (layer transfer experiment).
    """
    def __init__(self, gpt_model, n_layers_to_use=None, num_labels=2):
        super().__init__()
        self.gpt = gpt_model
        self.n_layers_to_use = n_layers_to_use  # None = use all layers
        self.classifier = nn.Linear(gpt_model.config.n_embd, num_labels)
        self.dropout = nn.Dropout(gpt_model.config.dropout)

    def forward(self, input_ids):
        B, T = input_ids.size()
        positions = torch.arange(T, device=input_ids.device).unsqueeze(0).expand(B, T)

        # Get embeddings
        x = self.gpt.dropout(
            self.gpt.token_embedding(input_ids) +
            self.gpt.position_embedding(positions)
        )

        # Apply only n_layers_to_use blocks
        blocks = self.gpt.blocks[:self.n_layers_to_use] if self.n_layers_to_use is not None else self.gpt.blocks
        for block in blocks:
            x = block(x)

        # Find [EOS] position for each item in batch and use that representation
        eos_mask = (input_ids == EXTRACT_IDX)
        eos_positions = eos_mask.long().argmax(dim=1)
        cls_output = x[torch.arange(B), eos_positions]

        return self.classifier(self.dropout(cls_output))

def run_finetune(model, n_layers, train_loader, valid_loader, device, epochs=5):
    """Run fine-tuning with n_layers transferred from pre-training."""
    clf = GPTClassifier(model, n_layers_to_use=n_layers).to(device)
    clf_optimizer = torch.optim.Adam(clf.parameters(), lr=6.25e-5)
    clf_criterion = nn.CrossEntropyLoss()

    best_acc = 0
    for epoch in range(1, epochs + 1):
        clf.train()
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            labels    = batch["labels"].to(device)
            clf_optimizer.zero_grad()

            # Supervised classification loss
            logits = clf(input_ids)
            clf_loss = clf_criterion(logits, labels)

            # Auxiliary language modeling loss (Section 3.2 of the paper)
            # The paper adds LM loss as auxiliary during fine-tuning
            # λ = 0.5 as used in the paper — helps preserve language understanding
            lm_logits, _ = clf.gpt(input_ids)
            lm_loss = criterion(lm_logits.view(-1, config.vocab_size), input_ids.view(-1))

            # Total loss = classification loss + λ * language modeling loss
            loss = clf_loss + 0.5 * lm_loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(clf.parameters(), 1.0)
            clf_optimizer.step()

        clf.eval()
        correct = total = 0
        with torch.no_grad():
            for batch in valid_loader:
                logits = clf(batch["input_ids"].to(device))
                preds  = logits.argmax(dim=-1)
                correct += (preds == batch["labels"].to(device)).sum().item()
                total   += batch["labels"].size(0)
        acc = correct / total
        if acc > best_acc:
            best_acc = acc

    return best_acc

# Load best pre-trained checkpoint
model.load_state_dict(torch.load("gpt_best.pt"))

print("Layer transfer experiment — mirroring Figure 2 left of the paper")
print("-" * 60)
layer_transfer_results = {}

for n_layers in [0, 1, 2, 3, 4]:
    label = "embeddings only" if n_layers == 0 else f"{n_layers} layer{'s' if n_layers > 1 else ''}"
    acc = run_finetune(model, n_layers if n_layers > 0 else None, train_sst2, valid_sst2, device)
    # Note: n_layers=0 means no transformer blocks, just embeddings
    # We handle this by passing n_layers_to_use=0 which will use no blocks
    layer_transfer_results[n_layers] = acc
    print(f"Transfer {label:<20} | SST-2 Val Acc: {acc:.3f}")

Layer transfer experiment — mirroring Figure 2 left of the paper
------------------------------------------------------------
Transfer embeddings only      | SST-2 Val Acc: 0.776
Transfer 1 layer              | SST-2 Val Acc: 0.784
Transfer 2 layers             | SST-2 Val Acc: 0.769
Transfer 3 layers             | SST-2 Val Acc: 0.773
Transfer 4 layers             | SST-2 Val Acc: 0.768


---
## Step 13 — Full fine-tuning result


In [18]:
# Full fine-tuning with all layers
model.load_state_dict(torch.load("gpt_best.pt"))
print("Full fine-tuning with all 4 layers...")
full_acc = run_finetune(model, None, train_sst2, valid_sst2, device, epochs=5)
print(f"GPT full fine-tuning SST-2 Val Acc: {full_acc:.3f}")

Full fine-tuning with all 4 layers...
GPT full fine-tuning SST-2 Val Acc: 0.783


---
## Step 14 — Results Summary


In [19]:
print("=" * 65)
print("GPT-1 RESULTS SUMMARY")
print("=" * 65)

print("\n1. ZERO-SHOT EVALUATION (Figure 2 right)")
print(f"{'Checkpoint':<15} {'Val Loss':<12} {'SST-2 Zero-shot'}")
for epoch, acc in sorted(zero_shot_results.items()):
    val_loss = checkpoints[epoch]
    print(f"Epoch {epoch:<9} {val_loss:<12.3f} {acc:.3f}")

print("\n2. LAYER TRANSFER EXPERIMENT (Figure 2 left)")
print(f"{'Layers transferred':<25} {'SST-2 Val Acc'}")
for n, acc in sorted(layer_transfer_results.items()):
    label = "0 (embeddings only)" if n == 0 else str(n)
    print(f"{label:<25} {acc:.3f}")

print(f"\n{'All 4 layers (full)':<25} {full_acc:.3f}")

GPT-1 RESULTS SUMMARY

1. ZERO-SHOT EVALUATION (Figure 2 right)
Checkpoint      Val Loss     SST-2 Zero-shot
Epoch 2         9.114        0.495
Epoch 4         7.593        0.495
Epoch 6         7.295        0.495
Epoch 8         6.960        0.495
Epoch 10        6.790        0.495

2. LAYER TRANSFER EXPERIMENT (Figure 2 left)
Layers transferred        SST-2 Val Acc
0 (embeddings only)       0.776
1                         0.784
2                         0.769
3                         0.773
4                         0.768

All 4 layers (full)       0.783
